<a href="https://colab.research.google.com/github/UbaidMalik365/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UbaidMalik365/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "/content/flyrank-ml-internship-starter"

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)

print("Current folder:", os.getcwd())
print("Dataset exists:", os.path.exists("data/raw/content_refresh_anonymized.csv"))

Current folder: /content/flyrank-ml-internship-starter
Dataset exists: True


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Method choice

I will use a Decision Tree classifier because my lane is about identifying pages that may need content review. A Decision Tree is suitable because it can learn simple if/else relationships between observable page signals and the declining label. It is also easy to interpret, which is important for decision-support.

I will use a small tree depth so that the model remains readable and does not become unnecessarily complex. I will compare the model with my Week-4 baseline using the same evaluation metric.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = df[features].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

y = df["is_declining_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training declining rate:", round(y_train.mean(), 3))
print("Testing declining rate:", round(y_test.mean(), 3))

Training rows: 24000
Testing rows: 6000
Training declining rate: 0.542
Testing declining rate: 0.542


## Split design

I will use an 80/20 train-test split. The model will be trained on 80% of the pages and evaluated on the remaining 20%.

This is an honest split because the test pages are not used to train the model. I will use the same target definition and observable features as the baseline. The test set will be used only for evaluation, so the comparison measures how well the model generalizes to unseen pages.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [18]:
from sklearn.tree import DecisionTreeClassifier

# Train a small, readable Decision Tree
tree = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

tree.fit(X_train, y_train)

# Model scores on unseen test pages
tree_scores = tree.predict_proba(X_test)[:, 1]

# Week-4 baseline score on the same test pages
baseline_scores = (
    X_test["impressions_90d"]
    * (X_test["days_since_last_update"] >= 180).astype(int)
)

# Precision@50 function
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    top_k = np.asarray(labels)[order[:k]]
    return top_k.mean()

baseline_p50 = precision_at_k(
    baseline_scores,
    y_test,
    50
)

tree_p50 = precision_at_k(
    tree_scores,
    y_test,
    50
)

comparison = pd.DataFrame({
    "Method": ["Week-4 Baseline", "Decision Tree"],
    "Precision@50": [baseline_p50, tree_p50]
})

display(comparison)

print("Baseline Precision@50:", round(baseline_p50, 3))
print("Decision Tree Precision@50:", round(tree_p50, 3))

,Method,Precision@50
0,Week-4 Baseline,0.46
1,Decision Tree,0.62


Baseline Precision@50: 0.46
Decision Tree Precision@50: 0.62


## Train + compare vs my baseline

I will train a depth-2 Decision Tree using the same observable features used in the task framing. I will evaluate both the model and the baseline on the same test set using Precision@50.

The baseline uses a simple stale-and-visible rule, while the Decision Tree can learn combinations of several signals. The goal is not to make the model as complex as possible, but to see whether it provides useful improvement over the transparent baseline.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [19]:
# Inspect the top 50 model-ranked pages and identify errors

test_results = X_test.copy()

test_results["actual_declining"] = y_test
test_results["tree_score"] = tree_scores

test_results["predicted_top50"] = 0

top50_indices = np.argsort(-tree_scores)[:50]
test_results.iloc[top50_indices, test_results.columns.get_loc("predicted_top50")] = 1

# False positives: ranked in top 50 but not actually declining
false_positives = test_results[
    (test_results["predicted_top50"] == 1) &
    (test_results["actual_declining"] == 0)
]

# False negatives: declining pages that were not ranked in top 50
false_negatives = test_results[
    (test_results["predicted_top50"] == 0) &
    (test_results["actual_declining"] == 1)
]

print("Top-50 false positives:", len(false_positives))
print("Declining pages outside top 50:", len(false_negatives))

print("\nTop model-ranked pages:")
display(
    test_results.sort_values("tree_score", ascending=False).head(10)
)

Top-50 false positives: 19
Declining pages outside top 50: 3221

Top model-ranked pages:


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,actual_declining,tree_score,predicted_top50
13891,92,20,1210,4.8,0.58,2652.0,0,0.60144,0
10359,95,20,720,1.3,0.00,2562.0,1,0.60144,0
22209,225,20,3299,24.2,0.06,3575.0,1,0.60144,1
11821,319,20,2326,2.8,1.93,1356.0,0,0.60144,1
14908,148,104,2084,6.1,0.14,0.0,0,0.60144,1
4256,331,104,112,10.4,0.89,1334.0,1,0.60144,1
27078,329,104,890,11.8,0.11,0.0,1,0.60144,1
28145,95,20,368,60.8,0.27,2762.0,0,0.60144,1
12174,329,104,49,18.2,0.00,0.0,1,0.60144,0
9024,223,104,677,19.3,0.30,6532.0,0,0.60144,1


## Errors and interpretation

The Decision Tree achieved a Precision@50 of 0.62 on the test set, compared with 0.46 for the Week-4 baseline. This means that 31 of the top 50 pages ranked by the tree were declining, compared with 23 of the top 50 pages ranked by the baseline.

The model appears to improve on the simple stale-and-visible rule by combining multiple observable signals rather than relying on only two conditions.

However, the model is not perfect. There were 19 false positives among the top 50 pages, and 3,221 declining pages were outside the top 50. This shows that the model is better suited for prioritizing pages for review than identifying every declining page.

The model also produced tied scores for several pages, so the exact order of pages with the same score should not be over-interpreted.

The result is directional and supports decision-making. It does not prove that the model will cause a page to improve after a refresh, and it does not establish a causal relationship with search engine rankings.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.